In [1]:
import os
from pathlib import Path

conda_prefix = Path(os.environ["CONDA_PREFIX"])
os.environ["R_HOME"] = str(conda_prefix / "lib" / "R")

import config_Elancet_FABEXIO
import fabio_functionsv2
import fabio_functionsv1_2

import numpy as np
import pandas as pd
import rpy2.robjects as ro

r_library = conda_prefix / "lib" / "R" / "library"
ro.r(f'.libPaths("{r_library}")')

pd.set_option("display.float_format", "{:.2e}".format)

Error importing in API mode: ImportError('libopenblas.so.0: cannot open shared object file: No such file or directory')
Trying to import in ABI mode.


# Basic scenario setting

In [2]:
year = 2017

In [3]:
# Single source of truth: EU region → country → ISO2
EU27_BY_REGION = {
    "North": {
        "Denmark": "DK",
        "Estonia": "EE",
        "Finland": "FI",
        "Ireland": "IE",
        "Latvia": "LV",
        "Lithuania": "LT",
        "Sweden": "SE",
    },
    "South": {
        "Croatia": "HR",
        "Cyprus": "CY",
        "Greece": "GR",
        "Italy": "IT",
        "Malta": "MT",
        "Portugal": "PT",
        "Slovenia": "SI",
        "Spain": "ES",
    },
    "West": {
        "Austria": "AT",
        "Belgium": "BE",
        "France": "FR",
        "Germany": "DE",
        "Luxembourg": "LU",
        "Netherlands": "NL",
    },
    "East": {
        "Bulgaria": "BG",
        "Czech Republic": "CZ",
        "Hungary": "HU",
        "Poland": "PL",
        "Romania": "RO",
        "Slovakia": "SK",
    },
}


# Region → country list
EU_regions = {
    region: list(countries)
    for region, countries in EU27_BY_REGION.items()
}


# Country → ISO2
eu27_iso2 = {
    country: iso2
    for countries in EU27_BY_REGION.values()
    for country, iso2 in countries.items()
}


# FABIO country names
EU27_fab = list(eu27_iso2)


# FOODTURE uses “Czechia” instead of “Czech Republic”
FOODTURE_NAME_MAP = {
    "Czech Republic": "Czechia",
}

EU27_f = [
    FOODTURE_NAME_MAP.get(country, country)
    for country in EU27_fab
]


In [4]:
regions_df = pd.read_csv(
    config_Elancet_FABEXIO.auxiliary_path / "fabio-exio_reg.csv",
    index_col=[0]
)

fabio_exio_supply_df = (
    pd.read_csv(
        config_Elancet_FABEXIO.auxiliary_path / "fabio-exio_sup.csv",
        header=[0],
        index_col=[0, 1, 2, 3]
    )
    .replace(0, np.nan)
    .rename_axis(["sector"], axis=1)
    .droplevel([0, 2, 3])
    .rename_axis(["item"], axis=0)
    .stack()
    .reset_index()
    .drop([0], axis=1)
)

## Calories & Biomass

In [ ]:
calories_path = config_Elancet_FABEXIO.calories_path

fabio_reader_calories = fabio_functionsv1_2.read(
    path=calories_path,
    year=f"{year}",
    version=1.2
)

fabio_items = fabio_reader_calories.items
fabio_regions = fabio_reader_calories.regions
fabio_io_codes = fabio_reader_calories.io_codes

Mcal = fabio_reader_calories.Y()


biomass_path = config_Elancet_FABEXIO.biomass_path

fabio_reader_biomass = fabio_functionsv1_2.read(
    path=biomass_path,
    year=f"{year}",
    version=2.0
)

fabio_items = fabio_reader_biomass.items
fabio_regions = fabio_reader_biomass.regions
fabio_io_codes = fabio_reader_biomass.io_codes

biomass = fabio_reader_biomass.Y()

In [6]:
biomass_food=(biomass
              .loc[:,lambda df:df.columns.get_level_values(5)=='food']
              .droplevel([0,2,4,5,6])
              .droplevel([0, 1,3,4,5],axis=1)
)


Mcal_food=(
    Mcal
    .loc[:,lambda df:df.columns.get_level_values(5)=='food']
    .droplevel([0,2,4,5,6])
    .droplevel([0,1,3,4,5],axis=1)
)

kcal_per_ton=(
    Mcal_food.groupby(level=1).sum().sum(axis=1)  
    .div(biomass_food.groupby(level=1).sum().sum(axis=1))
    .fillna(0)
    .div(1000)     # cal -> kcal
    .mul(1000000)  # million -> euro
).to_frame('kcal per ton')


## FABIO V2.0

In [ ]:
fabio_filepath = config_Elancet_FABEXIO.fabio_filepath

fabio_reader = fabio_functionsv2.read(
    path=fabio_filepath,
    year=f"{year}",
    version=2.0
)

fabio_items = fabio_reader.items
fabio_regions = fabio_reader.regions
fabio_io_codes = fabio_reader.io_labels

fabio_E = fabio_reader.E()
fabio_X = fabio_reader.X()
fabio_Y = fabio_reader.Y()

fabio_Z_mass = fabio_reader.Z(version="mass")


Zf = fabio_Z_mass.copy()
Yf = fabio_Y.copy()
Ef = fabio_E.copy()
Xf=fabio_X.copy()

In [8]:
Yf_iso3c = (
    Yf
    .loc[
        :,
        Yf.columns.isin(
            config_Elancet_FABEXIO.fabio_final_demand_categories,
            level="final demand"
        )
    ]
    .T.groupby(["iso3c"]).sum().T
)

if config_Elancet_FABEXIO.exclude_negatives:
    Yf_iso3c[Yf_iso3c < 0] = 0


xf = Yf_iso3c.sum(axis=1) + Zf.sum(axis=1)
Af = Zf.div(xf, axis=1).replace([np.nan, np.inf, -np.inf], 0)

A_issue = Af.loc[np.diag(Af >= 1), np.diag(Af >= 1)]
A_issues = (
    pd.concat(
        [
            pd.Series(
                np.diag(A_issue),
                index=A_issue.index
            ).to_frame("A_diagonal_value")
        ])
)
Af_modified = Af.copy()
for issue in A_issue.index:
    Af_modified.loc[issue, issue] = 1-1e-10


If = pd.DataFrame(
    np.diag(np.ones(len(Af))),
    index=Af.index,
    columns=Af.columns
)
Lf = pd.DataFrame(
    np.linalg.inv((If-Af_modified)),
    index=Af_modified.index,
    columns=Af_modified.index
)

if config_Elancet_FABEXIO.exclude_negatives:
    Lf[Lf < 0] = 0


Lf_no_grazing = Lf.drop(["Grazing"], level="item", axis=0)
xf_no_grazing = xf.drop(["Grazing"], level="item", axis=0)


# Supply mapping from extracted EXIOBASE sectors to FABIO
xf_mapped = (
    xf[
        xf
        .index
        .get_level_values("item")
        .isin(fabio_exio_supply_df["item"])
    ]
    .to_frame("biomass")
    .reset_index()
    .merge(
        fabio_exio_supply_df,
        on="item",
        how="left"
    )
    .merge(
        regions_df[["area", "region"]],
        left_on="area",
        right_on="area"
    )
    .pivot(
        columns=xf.index.names,
        index=["region", "sector"],
        values="biomass"
    )
)

In [9]:
M_exio_fabio = (
    xf_mapped
    .div(
        xf_mapped.sum(axis=1),
        axis=0)
    .replace([np.nan, np.inf, -np.inf], 0)
    .loc[lambda df:df.index.isin(config_Elancet_FABEXIO.extracted_sectors, level="sector")]
) 

M_exio_fabio_normalised = (
    M_exio_fabio
    .div(xf_no_grazing, axis=1)
    .replace([np.nan, np.inf, -np.inf], 0)
)


## EXIOBASE V3.9.6

In [10]:
exiobase_396path=config_Elancet_FABEXIO.fabio_filepath

Ze = pd.read_csv(
    exiobase_396path/f"IOT_{year}_pxp"/ "Z.txt",
    sep="\t",
    index_col=[0, 1],
    header=[0, 1]
)


Ae = pd.read_csv(
    exiobase_396path/f"IOT_{year}_pxp"/ "A.txt",
    sep="\t",
    index_col=[0, 1],
    header=[0, 1]
)

Ye = pd.read_csv(
    exiobase_396path/f"IOT_{year}_pxp"/"Y.txt",
    sep="\t",
    index_col=[0, 1],
    header=[0, 1]
)

xe = Ye.sum(axis=1) + Ze.sum(axis=1)

In [11]:
Fe_nutrients = pd.read_csv(
    exiobase_396path/f"IOT_{year}_pxp"/ "nutrients" / "F.txt",
    sep="\t",
    index_col=[0],
    header=[0, 1]
)

Fe_land = pd.read_csv(
    exiobase_396path/f"IOT_{year}_pxp"/ "land" / "F.txt",
    sep="\t",
    index_col=[0],
    header=[0, 1]
)

Fe_water = pd.read_csv(
    exiobase_396path/f"IOT_{year}_pxp"/ "water" / "F.txt",
    sep="\t",
    index_col=[0],
    header=[0, 1]
)

Fe_emission = pd.read_csv(
    exiobase_396path/f"IOT_{year}_pxp"/ "air_emissions" / "F.txt",
    sep="\t",
    index_col=[0],
    header=[0, 1]
)


In [12]:
Se_nutrient = (
    Fe_nutrients
    .div(xe)
    .replace([np.nan, np.inf, -np.inf], 0)
).sort_index()


Se_land = (
    Fe_land
    .div(xe)
    .replace([np.nan, np.inf, -np.inf], 0)
).sort_index()


Se_water = (
    Fe_water
    .div(xe)
    .replace([np.nan, np.inf, -np.inf], 0)
).sort_index()


Se_emission = (
    Fe_emission
    .div(xe)
    .replace([np.nan, np.inf, -np.inf], 0)
).sort_index()



Se=pd.concat([Se_nutrient,
              Se_land,
              Se_water,
              Se_emission])


## FABEXIO 

In [13]:
# Selecting different sub-matrices
Y1 = (
    Ye
    .loc[
        Ye.index.isin(config_Elancet_FABEXIO.extracted_sectors, level="sector"),
        :
    ]
)
Y2 = (
    Ye
    .loc[
        Ye.index.isin(config_Elancet_FABEXIO.other_sectors, level="sector"),
        :
    ]
)

In [14]:
A11 = (
    Ae
    .loc[
        Ae.index.isin(config_Elancet_FABEXIO.extracted_sectors, level="sector"),
        Ae.columns.isin(config_Elancet_FABEXIO.extracted_sectors, level="sector"),
    ]
)
A12 = (
    Ae
    .loc[
        Ae.index.isin(config_Elancet_FABEXIO.extracted_sectors, level="sector"),
        Ae.columns.isin(config_Elancet_FABEXIO.other_sectors, level="sector"),
    ]
)

A22 = (
    Ae
    .loc[
        Ae.index.isin(config_Elancet_FABEXIO.other_sectors, level="sector"),
        Ae.columns.isin(config_Elancet_FABEXIO.other_sectors, level="sector"),
    ]
)
A21 = (
    Ae
    .loc[
        Ae.index.isin(config_Elancet_FABEXIO.other_sectors, level="sector"),
        Ae.columns.isin(config_Elancet_FABEXIO.extracted_sectors, level="sector"),
    ]
)



In [15]:
# Calculating inverses
I11 = pd.DataFrame(
    np.eye(len(A11)),
    index=A11.index,
    columns=A11.columns,
)
I22 = pd.DataFrame(
    np.eye(len(A22)),
    index=A22.index,
    columns=A22.columns,
)

L22 = pd.DataFrame(
    np.linalg.pinv(I22-A22),
    index=A22.index,
    columns=A22.columns,
)

# Calculating HEM mulitpliers
# H11 is the same as H in the model documentation (and in the Edgar et al. paper)
# H11 = inv(I11 - A11 - A12.dot(L22.dot(A21)))

H11 = pd.DataFrame(
    np.linalg.inv(
        I11 - A11 - A12.dot(L22.dot(A21)) 
    ),
    index=A11.index,
    columns=A11.columns
)

# H21 = L22.dot(A21.dot(H11))
H21 = (
    L22.dot(A21.dot(H11))
)

# H12 = A12.dot(L22)
H12 = (
    A12.dot(L22)
)

# H22 = H21.dot(H12) = L22.dot(A21.dot(H11.dot(A12.dot(L22)))) 
#! This is really not needed as it needs to be diagonalised.
H22 = (
    H21.dot(H12)
)

# NOTE: H11 is the H from the Hertwich et al. paper
H = pd.concat([
    pd.concat([H11, H12], axis=1),
    pd.concat([H21, H22], axis=1),
], axis=0)


## Diagonalised demands on extracted sectors
# Direct demand
direct_demand_diagonal = pd.DataFrame(
    np.diag(Y1.sum(axis=1)),
    index=Y1.index,
    columns=Y1.index
)
# Indirect demand
indirect_demand_diagonal = pd.DataFrame(
    np.diag(H12.dot(Y2.sum(axis=1))),
    index=H12.index,
    columns=H12.index
)

## Upstream production

# Direct demand on extrated sectors and their upstream demand on extracted sectors.
sigma11 = H11.dot(direct_demand_diagonal)

# Indirect demand on extracted sectors and their upstream demand on extracted sectors. 
sigma12 = H11.dot(indirect_demand_diagonal)

# Direct demand on extracted sectors and their upstream demand on non-extracted sectors.
sigma21 = H21.dot(direct_demand_diagonal)

# Indirect demand on extracted sectors and their upstream demand on non-extracted sectors.
sigma22 = H21.dot(indirect_demand_diagonal)

### sigma

In [16]:
sigma = (
    pd.concat([
        pd.concat([sigma11, sigma12], axis=1),
        pd.concat([sigma21, sigma22], axis=1)
    ], axis=0)
    .T.groupby(["region", "sector"]).sum().T
)


In [17]:
group_map = {
    "Whole_grains": config_Elancet_FABEXIO.Whole_grains_fao,
    "Starchy_vegetables": config_Elancet_FABEXIO.Starchy_vegetables_fao,
    "Vegetables": config_Elancet_FABEXIO.Vegetables_fao,
    "Fruits": config_Elancet_FABEXIO.Fruits_fao,
    "Dairy_foods": config_Elancet_FABEXIO.Dairy_foods_fao,
    "Beef_lamb_pig": config_Elancet_FABEXIO.Beef_lamb_pig_fao,
    "Poultry": config_Elancet_FABEXIO.Poultry_fao,
    "Eggs": config_Elancet_FABEXIO.Eggs_fao,
    "Fish": config_Elancet_FABEXIO.Fish_fabio,
    "Legumes": config_Elancet_FABEXIO.Legumes_fao,
    "Nuts": config_Elancet_FABEXIO.Nuts_fao,
    "Unsaturated_oil": config_Elancet_FABEXIO.Unsaturated_oil_fao,
    "Palm_coco_oil": config_Elancet_FABEXIO.Palm_coco_oil_fao,
    "Lard_tallow_butter": config_Elancet_FABEXIO.Lard_tallow_butter_fao,
    "Sugars": config_Elancet_FABEXIO.Sugars_fao,
}

def aggregate_by_food_group(source_df, group_map, group_level_name="Food_group"):

    pieces = []
    keys = []

    for group_name, item_list in group_map.items():
        
        subset = source_df.loc[source_df.index.isin(item_list)]
        pieces.append(subset)
        keys.append(group_name)

    out = pd.concat(pieces, axis=0, keys=keys, names=[group_level_name])
    return out

In [18]:
# Population of EU27 in Year 2017-2020

pop_2015_2020 = (
    pd.read_csv(
        "population from 2015 to 2020.csv",
        index_col=[3,9]
    )[['Value']]
    .unstack()
    .droplevel(0,axis=1)
    .rename({'Netherlands (Kingdom of the)':'Netherlands'})
    .rename({'Czechia':'Czech Republic'})
    *(1e+3)
)

pop = (pop_2015_2020.iloc[:,-4:]).mean(axis=1)

# Population for 27 countries by region
pop_f=pop.to_frame('pop').T

country_to_region = {
    country: region
    for region, countries in EU_regions.items()
    for country in countries
}
regions = [country_to_region.get(col, "Unknown") for col in pop_f.columns]
pop_f.columns = pd.MultiIndex.from_arrays([regions, pop_f.columns],names=["EUregion", "Country"])
pop_f =pop_f.sort_index(axis=1)

# Population for four regions
pop_f_reg=pop_f.loc['pop'].groupby(level=0).sum()

## FOODTURE

The workflow below converts FOODTURE consumption data to FABIO items, fills
group-level missing observations from population-weighted regional donor
profiles, applies FABIO sourcing shares, and constructs the current-diet and
Planetary Health Diet final-demand matrices.


### Load and harmonize FOODTURE data

In [19]:
Summary_L7 = pd.read_excel(
    "Foodture_results.xlsx",
    sheet_name="Summary_L7",
    index_col=[0, 1, 2],
    header=0,
).fillna(0)

L7_meat_split = pd.read_excel(
    "Foodture_results.xlsx",
    sheet_name="L7_meat_split",
    index_col=[0, 1],
    header=0,
).fillna(0)
 

In [20]:
# FABIO food final demand for the EU27 (tonnes/year)
Yf_eu_fabio_yearly = (
    Yf.loc[:, Yf.columns.get_level_values("EU27")]
    .loc[:, lambda df: df.columns.get_level_values("final demand") == "food"]
    .droplevel([0, 1, 3, 4, 5, 7, 8, 9])
    .droplevel([0, 1, 3, 4, 5, 6], axis=1)
)

missing_pop = pd.Index(Yf_eu_fabio_yearly.columns).difference(pop.index)
if not missing_pop.empty:
    raise ValueError(f"Missing population for FABIO countries: {missing_pop.tolist()}")

# g/capita/day
Yf_eu_fabio_percapita = (
    Yf_eu_fabio_yearly
    .div(pop, axis=1)
    .div(365)
    .mul(1e6)
)


def meat_shares(items):
    totals = (
        Yf_eu_fabio_yearly
        .groupby(level="item")
        .sum()
        .reindex(items)
        .fillna(0)
    )
    shares = totals.div(totals.sum(axis=0).replace(0, np.nan), axis=1)
    if shares.isna().any().any():
        countries = shares.columns[shares.isna().any()].tolist()
        raise ValueError(f"Cannot calculate meat shares for: {countries}")
    return shares


Share_Bovine_pig = meat_shares(["Bovine Meat", "Pigmeat"])
Share_nation_meat = meat_shares([
    "Bovine Meat",
    "Mutton & Goat Meat",
    "Pigmeat",
    "Poultry Meat",
    "Meat, Other",
])


In [21]:
L7_meat_split = L7_meat_split.rename(index={"Czechia": "Czech Republic"})


def allocate_composite_meat(split_label, shares):
    """Allocate composite meat for modeled EU27 countries only."""

    selected = (
        L7_meat_split.loc[
            L7_meat_split.index.get_level_values(1) == split_label
        ]
        .droplevel(1)
    )

    # Exclude Norway, UK and other non-modelled countries
    selected = selected.loc[
        selected.index.isin(EU27_fab)
    ]

    amounts = selected["g/capita_rev"]
    allocation = shares.T.reindex(selected.index)

    # Only check missing shares within the modeled EU27
    missing = allocation.index[
        allocation.isna().any(axis=1)
    ]

    if len(missing):
        raise ValueError(
            f"Missing FABIO meat shares for modeled EU27 countries: "
            f"{missing.tolist()}"
        )

    return (
        allocation
        .mul(amounts, axis=0)
        .stack()
        .rename("g/capita_rev")
        .to_frame()
    )

meat_allocated = pd.concat([
    allocate_composite_meat("split bovine pig", Share_Bovine_pig),
    allocate_composite_meat("split by national share", Share_nation_meat),
]).groupby(level=[0, 1]).sum()

L7_meat_split_sum = pd.concat(
    {"Meat": meat_allocated},
    names=[Summary_L7.index.names[0]],
)

L7_tot = (
    pd.concat([Summary_L7, L7_meat_split_sum])
    .groupby(level=[0, 1, 2])
    .sum()
)

L7_tot_simp = (
    L7_tot.groupby(level=[1, 2])
    .sum()
    .loc[EU27_f]
    .rename(index={"Czechia": "Czech Republic"}, level=0)
    .rename(index={"Cocoa and products": "Cocoa Beans and products"}, level=1)
    .rename(columns={"g/capita_rev": "Foodture"})
)

fabio_daily = (
    Yf_eu_fabio_percapita.groupby(level="item").sum()
    .T.stack()
    .rename("FABIO")
)

Yf_Foodture_FABIO_summary = pd.concat(
    [fabio_daily, L7_tot_simp["Foodture"]],
    axis=1,
)
Adult_Survey_fabio_ori = Yf_Foodture_FABIO_summary[["Foodture"]].fillna(0)


In [22]:
Adult_Survey_fabio_ori = (
    Yf_Foodture_FABIO_summary[["Foodture"]]
    .fillna(0)
)

# Only rename index levels; values remain unchanged
Adult_Survey_fabio_ori.index = (
    Adult_Survey_fabio_ori.index
    .set_names(["area", "item"])
)


In [23]:
Adult_Survey_fabio_ori

Foodture
area    item                         
Austria Abaca                0.00e+00
        Alcohol, Non-Food    0.00e+00
        Apples and products  1.05e+02
        Asses                0.00e+00
        Bananas              2.13e+01
...                               ...
Sweden  Vegetables, Other    1.72e+02
        Wheat and products   2.01e+02
        Wine                 0.00e+00
        Wool (Clean Eq.)     0.00e+00
        Yams                 0.00e+00

[3321 rows x 1 columns]

### Fill missing FOODTURE food groups


In [24]:
IMPUTE_GROUPS = ("Legumes", "Nuts", "Palm_coco_oil")


def impute_group_from_region(survey, groups=IMPUTE_GROUPS):
    """
    Replace a country group's all-zero profile with its regional profile.

    The regional item profile is population-weighted and excludes countries
    whose entire group is missing; including their zero placeholders would
    bias the regional mean downward.
    """
    wide = survey["Foodture"].unstack(level="item").copy()
    records = []

    for group in groups:
        group_items = pd.Index(group_map[group]).intersection(wide.columns)
        missing_group = wide[group_items].sum(axis=1).eq(0)

        for region, region_countries in EU_regions.items():
            region_countries = pd.Index(region_countries).intersection(wide.index)
            recipients = region_countries[missing_group.reindex(region_countries)]
            if recipients.empty:
                continue

            donors = region_countries[~missing_group.reindex(region_countries)]
            if donors.empty:
                raise ValueError(f"No donor country for {group} in {region}")

            donor_pop = pop.reindex(donors)
            regional_profile = (
                wide.loc[donors, group_items]
                .mul(donor_pop, axis=0)
                .sum(axis=0)
                .div(donor_pop.sum())
            )
            wide.loc[recipients, group_items] = regional_profile.to_numpy()

            records.extend(
                {"Country": country, "Food_group": group, "Donor_region": region}
                for country in recipients
            )

    filled = wide.stack().rename("Foodture").to_frame().reindex(survey.index)
    report = pd.DataFrame(records, columns=["Country", "Food_group", "Donor_region"])
    return filled, report


Adult_Survey_fabio, imputation_report = impute_group_from_region(
    Adult_Survey_fabio_ori
)

In [25]:
remaining_missing = (
    aggregate_by_food_group(
        Adult_Survey_fabio["Foodture"].unstack(level=0),
        group_map,
    )
    .groupby(level="Food_group")
    .sum()
    .loc[list(IMPUTE_GROUPS)]
    .eq(0)
)

if remaining_missing.any().any():
    raise ValueError("At least one imputed food group is still zero")

### Apply FABIO sourcing shares and construct CD/PHD demand


In [26]:
# Source-country shares within each FABIO item and consuming country
Yf_eu_fabio_percapita_sum=(
    Yf_eu_fabio_percapita.groupby(level=1).sum().T.stack())

# %
Yf_eu_fabio_per=(
    Yf_eu_fabio_percapita.stack()
    .reorder_levels([2,1,0]).unstack()
    .div(Yf_eu_fabio_percapita_sum,
         axis=0)
).fillna(0)

# Yf_eu_foodture with FABIO sourcing information:
Yf_eu_diet_percapita_f=(
    Yf_eu_fabio_per.mul(Adult_Survey_fabio.copy()['Foodture'],axis=0)
)
Yf_eu_diet_percapita_f=(
    Yf_eu_diet_percapita_f.stack().reorder_levels([2,1,0]).unstack()
)

Yf_eu_diet_percapita_f_ton=(Yf_eu_diet_percapita_f.mul(1e-6))  

## Eat-lancet 

In [27]:
eat_lancet_df = (
    pd.Series({
        "Whole_grains":210,
        "Starchy_vegetables":50,
        "Vegetables":300,
        "Fruits":200,
        "Nuts":50,
        "Legumes":75,
        "Dairy_foods":250,
        "Poultry":30,
        "Fish":30,
        "Eggs":15,
        "Beef_lamb_pig":15,
        "Unsaturated_oil":40,
        "Palm_coco_oil":6,
        "Lard_tallow_butter":5,
        "Sugars":30})
).to_frame(name="EAT-Lancet g/day")

eat_lancet_df.index.name = 'Food_group'


## Y current diet (group map)

In [28]:
Y_curdiet_general = aggregate_by_food_group(
    (Yf_eu_diet_percapita_f.groupby(level=1).sum()),
    group_map,
    group_level_name="Food_group"
) # Current diet with PHD category (unit: g)

ratio_Y = 1 / (
    Y_curdiet_general.groupby(level=0).sum()
    .div(eat_lancet_df['EAT-Lancet g/day'], axis=0)
)

In [29]:
Y_curdiet_trade = aggregate_by_food_group(
    (Yf_eu_diet_percapita_f_ton
        .reorder_levels([1,0]).sort_index().unstack()),
    group_map,
    group_level_name="Food_group"
) # show subfood categories and supplychain

Y_curdiet_trade_r=(
    Y_curdiet_trade
        .stack()
        .stack()
        .reorder_levels([3,0,1,2])
        .unstack()
        .unstack()
) # only show subfood categories


ratio_S = ratio_Y.T.stack()
Y_phdiet_trade_r = Y_curdiet_trade_r.mul(ratio_S, axis=0)

Y_phdiet_trade_perton=(
    Y_phdiet_trade_r.T.stack()
        .groupby(level=[0,1])
        .sum()
        .reindex(Yf_eu_diet_percapita_f_ton.index)
        .fillna(0)
)

In [30]:
Y_phdiet_trade_perton.sum()

area
Austria          1.31e-03
Belgium          1.31e-03
Bulgaria         1.31e-03
Croatia          1.31e-03
Cyprus           1.23e-03
Czech Republic   1.31e-03
Denmark          1.31e-03
Estonia          1.31e-03
Finland          1.31e-03
France           1.31e-03
Germany          1.31e-03
Greece           1.31e-03
Hungary          1.31e-03
Ireland          1.31e-03
Italy            1.31e-03
Latvia           1.30e-03
Lithuania        1.31e-03
Luxembourg       1.18e-03
Malta            1.23e-03
Netherlands      1.31e-03
Poland           1.31e-03
Portugal         1.23e-03
Romania          1.31e-03
Slovakia         1.31e-03
Slovenia         1.31e-03
Spain            1.31e-03
Sweden           1.31e-03
dtype: float64

# GLAM 

In [31]:
GLAM_extra_agg = (
    pd.read_csv(
        config_Elancet_FABEXIO.GLAM_extra_path / "glam_impacts_D_pba.tsv",
        sep="\t",
        header=0,
        index_col=[0, 1, 2, 3, 4, 5],
        engine="python",
    )
    .loc[year]
)

GLAM_PATHWAYS = {
    "Climate Change TR": ("EQ Climate Change-TR", "Terrestrial"),
    "Land Use TR": ("EQ Land use", "Terrestrial"),
    "Terrestrial Acidification TR": ("EQ Terrestrial Acidification", "Terrestrial"),
    "Terrestrial Ecotoxicicity TR": ("EQ Ecotoxcicity-TR", "Terrestrial"),
    "Climate Change FW": ("EQ Climate Change-FR", "Freshwater"),
    "Water Consumption FW": ("EQ Water Consumption", "Freshwater"),
    "Freshwater Ecotoxicicity FW": ("EQ Ecotoxcicity-FR", "Freshwater"),
    "Freshwater Eutrophication FW": ("EQ Freshwater Eutrophication", "Freshwater"),
}


def extract_glam_pathway(method_name, realm):
    index = GLAM_extra_agg.index
    mask = (
        (index.get_level_values("LCIAMethod_name") == method_name)
        & (index.get_level_values("LCIAMethod_realm") == realm)
        & (index.get_level_values("LCIAMethod_type") == "Damage level")
    )
    selected = GLAM_extra_agg.loc[mask]
    if selected.empty:
        raise ValueError(f"No GLAM factors for {method_name} / {realm}")
    return selected.droplevel([
        "LCIAMethod_name",
        "LCIAMethod_realm",
        "LCIAMethod_type",
    ])["value"]


F_sum = pd.DataFrame({
    label: extract_glam_pathway(method, realm)
    for label, (method, realm) in GLAM_PATHWAYS.items()
}).T

CF_sum = (
    F_sum.div(xe, axis=1)
    .replace([np.nan, np.inf, -np.inf], 0)
    .reindex(columns=sigma.index, fill_value=0)
)


# Biodiversity calculation

In [32]:
Lf_version=(
    Lf_no_grazing
    .droplevel([0,1,3,4,5,7,8,9])
    .droplevel([0,1,3,4,5,7,8,9], axis=1)
)

M2 = (
    M_exio_fabio_normalised.loc[
        :,
        lambda df: ~(df.columns.get_level_values("item") == "Grazing")
        ]
    .droplevel([0,1,3,4,5,7,8,9], axis=1)
    .loc[:, Lf_version.index]
)
    
upstream_activities = (
    sigma.dot(
        M2
        .dot(
            Lf_version.dot(
                Yf_eu_diet_percapita_f_ton
            )
        )
    )
)

In [33]:
# Biodiversity impact splited by impact categories

Bio_sum = {}

for impact in CF_sum.index:
    Bio_sum[impact] = upstream_activities.mul(
        CF_sum.loc[impact].T, axis=0
    )

Bio_sum_pdf = pd.concat(Bio_sum, names=['impacts'])



# 
Bio_sum_pdf_r= Bio_sum_pdf.copy()
country_to_region = {
    country: region
    for region, countries in EU_regions.items()
    for country in countries
}


regions = [country_to_region.get(col, "Unknown") for col in Bio_sum_pdf_r.columns]

Bio_sum_pdf_r.columns = pd.MultiIndex.from_arrays([regions, Bio_sum_pdf_r.columns], 
                                                  names=["EUregion", "Country"])
Bio_sum_pdf_r=Bio_sum_pdf_r.sort_index(axis=1)

In [34]:
# Biodiversity impact splited by FABIO food categories

CSSigma_sum = {}

for item in CF_sum.index:
    CSSigma_sum[item] = sigma.mul(
        CF_sum.loc[item].T, axis=0
    )
    
CSSigma = pd.concat(CSSigma_sum, names=['impact_type'])
CSSigma_rev=CSSigma.groupby(level=0).sum() 

CSSigma_rev_TR=CSSigma_rev.loc[lambda df:df.index.get_level_values(0).str.contains('TR')].sum().to_frame('TR').T
Left_TR=CSSigma_rev_TR.loc[['TR']].dot(M2).dot(Lf_version)

CSSigma_rev_FW=CSSigma_rev.loc[lambda df:df.index.get_level_values(0).str.contains('FW')].sum().to_frame('FW').T
Left_FW=CSSigma_rev_FW.loc[['FW']].dot(M2).dot(Lf_version)

# CD
IMP_TRcd = {}

for land in Yf_eu_diet_percapita_f_ton.columns:
    y_vec = Yf_eu_diet_percapita_f_ton[land]  # shape (n,)

    y_diag_cd = pd.DataFrame(
        np.diag(y_vec),
        index=Yf_eu_diet_percapita_f_ton.index,
        columns=Yf_eu_diet_percapita_f_ton.index
    )  # shape (n, n)

    IMP_TRcd[land] = Left_TR.dot(y_diag_cd)

IMP_TR_df_cd = pd.concat(IMP_TRcd, axis=1).stack().stack()

IMP_TR_cd=IMP_TR_df_cd.groupby(level=1).sum()




IMP_FWcd = {}

for land in Yf_eu_diet_percapita_f_ton.columns:
    y_vec = Yf_eu_diet_percapita_f_ton[land]  # shape (n,)

    y_diag_cd = pd.DataFrame(
        np.diag(y_vec),
        index=Yf_eu_diet_percapita_f_ton.index,
        columns=Yf_eu_diet_percapita_f_ton.index
    )  # shape (n, n)

    IMP_FWcd[land] = Left_FW.dot(y_diag_cd)

IMP_FW_df_cd = pd.concat(IMP_FWcd, axis=1).stack().stack()

IMP_FW_cd=IMP_FW_df_cd.groupby(level=1).sum()

In [35]:
# PHD 

IMP_TRphd = {}

for land in Y_phdiet_trade_perton.columns:
    y_vec = Y_phdiet_trade_perton[land]  

    y_diag = pd.DataFrame(
        np.diag(y_vec),
        index=Y_phdiet_trade_perton.index,
        columns=Y_phdiet_trade_perton.index
    )  # shape (n, n)

    IMP_TRphd[land] = Left_TR.dot(y_diag)

IMP_TR_df_phd = pd.concat(IMP_TRphd, axis=1).stack().stack()

IMP_TR_phd=IMP_TR_df_phd.groupby(level=1).sum()

IMP_FWphd = {}

for land in Y_phdiet_trade_perton.columns:
    y_vec = Y_phdiet_trade_perton[land]  

    y_diag = pd.DataFrame(
        np.diag(y_vec),
        index=Y_phdiet_trade_perton.index,
        columns=Y_phdiet_trade_perton.index
    )  #

    IMP_FWphd[land] = Left_FW.dot(y_diag)

IMP_FW_df_phd = pd.concat(IMP_FWphd, axis=1).stack().stack()

IMP_FW_phd=IMP_FW_df_phd.groupby(level=1).sum()

In [36]:
IMP_FW_phd.sum()

Austria          2.16e-14
Belgium          3.25e-14
Bulgaria         3.31e-14
Croatia          2.36e-14
Cyprus           3.98e-14
Czech Republic   2.25e-14
Denmark          1.90e-14
Estonia          1.94e-14
Finland          2.01e-14
France           2.21e-14
Germany          2.68e-14
Greece           1.16e-13
Hungary          1.64e-14
Ireland          1.89e-14
Italy            3.52e-14
Latvia           1.79e-14
Lithuania        2.16e-14
Luxembourg       2.90e-14
Malta            4.99e-14
Netherlands      3.28e-14
Poland           1.57e-14
Portugal         4.82e-14
Romania          1.70e-14
Slovakia         2.43e-14
Slovenia         3.77e-14
Spain            4.79e-14
Sweden           2.35e-14
dtype: float64

In [37]:
IMP_TR_cd.sum()

Austria          1.33e-14
Belgium          1.73e-14
Bulgaria         1.87e-14
Croatia          1.25e-14
Cyprus           3.11e-14
Czech Republic   1.63e-14
Denmark          1.19e-14
Estonia          9.54e-15
Finland          1.20e-14
France           1.99e-14
Germany          1.43e-14
Greece           3.57e-14
Hungary          1.35e-14
Ireland          9.54e-15
Italy            2.24e-14
Latvia           1.23e-14
Lithuania        1.56e-14
Luxembourg       1.78e-14
Malta            2.55e-14
Netherlands      1.84e-14
Poland           1.14e-14
Portugal         4.12e-14
Romania          1.02e-14
Slovakia         1.82e-14
Slovenia         1.35e-14
Spain            2.25e-14
Sweden           1.53e-14
dtype: float64

In [38]:
IMP_FW_cd

,Austria,Belgium,Bulgaria,Croatia,Cyprus,Czech Republic,Denmark,Estonia,Finland,France,...,Luxembourg,Malta,Netherlands,Poland,Portugal,Romania,Slovakia,Slovenia,Spain,Sweden
item,,,,,,,,,,,,,,,,,,,,,
Abaca,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,...,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00
"Alcohol, Non-Food",0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,...,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00
Apples and products,1.48e-15,7.51e-16,7.47e-16,3.25e-16,6.61e-16,4.23e-16,1.89e-15,1.20e-15,5.01e-16,2.49e-16,...,8.81e-16,3.19e-16,6.68e-16,2.02e-16,7.62e-16,2.55e-16,4.84e-16,1.65e-15,8.57e-16,7.55e-16
Asses,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,...,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00
Bananas,1.81e-16,1.77e-16,3.55e-16,2.52e-16,1.63e-16,1.40e-16,2.70e-16,2.67e-16,2.42e-16,1.03e-16,...,1.82e-16,1.52e-16,2.73e-16,1.43e-16,4.33e-16,1.50e-16,1.40e-16,2.17e-16,0.00e+00,1.39e-16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
"Vegetables, Other",1.12e-15,1.63e-15,3.74e-15,1.11e-15,2.75e-15,8.01e-16,1.44e-15,1.10e-15,1.89e-15,1.73e-15,...,1.63e-15,6.94e-15,1.33e-15,5.47e-16,3.90e-15,6.31e-16,7.48e-16,2.21e-15,3.96e-15,2.78e-15
Wheat and products,6.40e-16,1.55e-15,9.36e-16,5.63e-16,1.94e-15,4.78e-16,2.69e-16,1.74e-16,1.86e-16,9.58e-16,...,2.07e-15,1.87e-15,1.07e-15,4.01e-16,1.58e-15,4.93e-16,6.52e-16,1.10e-15,1.78e-15,1.14e-15
Wine,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,...,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00


In [39]:
IMP_FW_phd

,Austria,Belgium,Bulgaria,Croatia,Cyprus,Czech Republic,Denmark,Estonia,Finland,France,...,Luxembourg,Malta,Netherlands,Poland,Portugal,Romania,Slovakia,Slovenia,Spain,Sweden
item,,,,,,,,,,,,,,,,,,,,,
Abaca,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,...,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00
"Alcohol, Non-Food",0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,...,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00
Apples and products,1.18e-15,5.82e-16,8.77e-16,3.95e-16,7.31e-16,4.92e-16,8.11e-16,7.50e-16,2.49e-16,1.70e-16,...,6.82e-16,3.20e-16,4.46e-16,2.07e-16,8.00e-16,3.30e-16,5.63e-16,1.53e-15,6.68e-16,4.75e-16
Asses,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,...,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00
Bananas,1.44e-16,1.37e-16,4.17e-16,3.06e-16,1.80e-16,1.63e-16,1.15e-16,1.67e-16,1.21e-16,7.00e-17,...,1.41e-16,1.52e-16,1.82e-16,1.47e-16,4.54e-16,1.95e-16,1.63e-16,2.02e-16,0.00e+00,8.76e-17
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
"Vegetables, Other",2.25e-15,2.90e-15,3.56e-15,1.59e-15,2.70e-15,2.28e-15,2.44e-15,1.85e-15,2.47e-15,2.12e-15,...,2.90e-15,6.82e-15,3.17e-15,1.01e-15,4.07e-15,1.15e-15,2.13e-15,3.94e-15,5.08e-15,3.49e-15
Wheat and products,7.57e-16,1.44e-15,1.36e-15,1.10e-15,2.59e-15,4.76e-16,3.09e-16,2.95e-16,3.54e-16,9.41e-16,...,2.03e-15,2.55e-15,1.21e-15,4.26e-16,1.70e-15,5.30e-16,6.48e-16,1.88e-15,2.53e-15,1.06e-15
Wine,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,...,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00,0.00e+00


# Calories (group)

In [40]:
# Kcal for CD

Energy_curdiet_per_capita=(Yf_eu_diet_percapita_f_ton
                           .groupby(level=1).sum().sort_index()
                           .mul(kcal_per_ton['kcal per ton'],axis=0)
                          )

Energy_curdiet_per_capita_d=(
    Yf_eu_diet_percapita_f_ton
    .reorder_levels([1,0])
    .unstack().mul(kcal_per_ton['kcal per ton'],axis=0)
    .stack().reorder_levels([1,0])
    .sort_index()
)

B_Energy_list_agg_general=(
    aggregate_by_food_group(
        (Energy_curdiet_per_capita),
        group_map,
        group_level_name="Food_group")
    .groupby(level=0).sum()
)



In [41]:
# include new level of region:

country_to_region = {
    country: region
    for region, countries in EU_regions.items()
    for country in countries
}

regions = [country_to_region.get(col, "Unknown") for col in B_Energy_list_agg_general.columns]

B_eng_with_region = B_Energy_list_agg_general.copy()
B_eng_with_region.columns = pd.MultiIndex.from_arrays([regions, B_eng_with_region.columns], 
                                                  names=["EUregion", "Country"])
B_eng_with_region=B_eng_with_region.sort_index(axis=1)


In [42]:
# Kcal for PH diet in EU27
B_Energy_ph = B_Energy_list_agg_general.mul(ratio_Y)

B_Energy_ph_region = B_Energy_ph.copy()
B_Energy_ph_region.columns = pd.MultiIndex.from_arrays([regions, B_Energy_ph.columns], 
                                                  names=["EUregion", "Country"])
B_Energy_ph_region=B_Energy_ph_region.sort_index(axis=1)


In [43]:
B_eng_with_region.sum()

EUregion  Country       
East      Bulgaria         2.11e+03
          Czech Republic   2.36e+03
          Hungary          2.95e+03
          Poland           2.38e+03
          Romania          2.11e+03
          Slovakia         2.40e+03
North     Denmark          2.28e+03
          Estonia          2.03e+03
          Finland          2.19e+03
          Ireland          2.17e+03
          Latvia           2.20e+03
          Lithuania        2.36e+03
          Sweden           2.88e+03
South     Croatia          1.78e+03
          Cyprus           2.44e+03
          Greece           2.47e+03
          Italy            2.43e+03
          Malta            2.42e+03
          Portugal         2.66e+03
          Slovenia         1.66e+03
          Spain            2.14e+03
West      Austria          2.04e+03
          Belgium          2.32e+03
          France           2.67e+03
          Germany          1.79e+03
          Luxembourg       2.23e+03
          Netherlands      2.36e+03
dty

# Environmental footprints calculation

In [44]:
# land use & water use & Freshwater eutrophication

Se_cropland = (
    Se.loc[Se.index.str.contains("Cropland", case=False, na=False)]
    .sum()
    .to_frame("Cropland")
    .T
)

Se_pasture = (
    Se.loc[Se.index.str.contains("pastures", case=False, na=False)]
    .sum()
    .to_frame("Pasture")
    .T
)

# Include forest, other land and infrastructure land
Se_otherland = (
    Se.loc[
        Se.index.str.contains(
            "Other land Use",
            case=False,
            regex=True,
            na=False
        )
    ]
    .sum()
    .to_frame("Other land")
    .T
)

Se_wateruse = (
    Se.loc[
        Se.index.str.contains(
            "Water Consumption Blue",
            case=False,
            na=False
        )
    ]
    .sum()
    .to_frame("Water use")
    .T
)

Se_FE_P = (
    Se.loc[
        Se.index.isin([
            "P - agriculture - water",
            "P - waste - water"
        ])
    ]
    .sum()
    .to_frame("FE-P")
    .T
)

Se_FE_N = (
    Se.loc[
        Se.index.isin([
            "N - agriculture - water",
            "N - waste - water"
        ])
    ]
    .sum()
    .to_frame("FE-N")
    .T
)

Se_sum = pd.concat([
    Se_cropland,
    Se_pasture,
    Se_otherland,
    Se_wateruse,
    Se_FE_P,
    Se_FE_N
])



# Se × sigma


SeSigma = pd.concat(
    {
        item: sigma.mul(Se_sum.loc[item], axis=0)
        for item in Se_sum.index
    },
    names=["impact_type"]
)

SeSigma_rev = SeSigma.groupby(level=0).sum()


SeSigma_lu_rev = (
    SeSigma_rev.loc[["Cropland", "Pasture", "Other land"]]
    .sum()
    .to_frame("Land use")
    .T
)

SeSigma_wu_rev = SeSigma_rev.loc[["Water use"]]
SeSigma_feN_rev = SeSigma_rev.loc[["FE-N"]]
SeSigma_feP_rev = SeSigma_rev.loc[["FE-P"]]



# Calculate footprints


Left_lu = SeSigma_lu_rev.dot(M2).dot(Lf_version)
Left_wu = SeSigma_wu_rev.dot(M2).dot(Lf_version)
Left_feN = SeSigma_feN_rev.dot(M2).dot(Lf_version)
Left_feP = SeSigma_feP_rev.dot(M2).dot(Lf_version)



FT_lu_grouped = (
    Yf_eu_diet_percapita_f_ton
    .mul(Left_lu.iloc[0], axis=0)
    .groupby(level=1)
    .sum()
)

FT_wu_grouped = (
    Yf_eu_diet_percapita_f_ton
    .mul(Left_wu.iloc[0], axis=0)
    .groupby(level=1)
    .sum()
)

FT_feN_grouped = (
    Yf_eu_diet_percapita_f_ton
    .mul(Left_feN.iloc[0], axis=0)
    .groupby(level=1)
    .sum()
)

FT_feP_grouped = (
    Yf_eu_diet_percapita_f_ton
    .mul(Left_feP.iloc[0], axis=0)
    .groupby(level=1)
    .sum()
)

# Add EU region level

country_to_region = {
    country: region
    for region, countries in EU_regions.items()
    for country in countries
}

regions = [
    country_to_region.get(country, "Unknown")
    for country in FT_lu_grouped.columns
]

region_columns = pd.MultiIndex.from_arrays(
    [regions, FT_lu_grouped.columns],
    names=["EUregion", "Country"]
)

FT_lu_grouped_r = FT_lu_grouped.copy()
FT_lu_grouped_r.columns = region_columns
FT_lu_grouped_r = FT_lu_grouped_r.sort_index(axis=1)

FT_wu_grouped_r = FT_wu_grouped.copy()
FT_wu_grouped_r.columns = region_columns
FT_wu_grouped_r = FT_wu_grouped_r.sort_index(axis=1)

FT_feN_grouped_r = FT_feN_grouped.copy()
FT_feN_grouped_r.columns = region_columns
FT_feN_grouped_r = FT_feN_grouped_r.sort_index(axis=1)

FT_feP_grouped_r = FT_feP_grouped.copy()
FT_feP_grouped_r.columns = region_columns
FT_feP_grouped_r = FT_feP_grouped_r.sort_index(axis=1)




FT_lu_grouped_r_general = (
    aggregate_by_food_group(
        FT_lu_grouped_r,
        group_map,
        group_level_name="Food_group"
    )
    .groupby(level=0)
    .sum()
)

FT_wu_grouped_r_general = (
    aggregate_by_food_group(
        FT_wu_grouped_r,
        group_map,
        group_level_name="Food_group"
    )
    .groupby(level=0)
    .sum()
)

FT_feN_grouped_r_general = (
    aggregate_by_food_group(
        FT_feN_grouped_r,
        group_map,
        group_level_name="Food_group"
    )
    .groupby(level=0)
    .sum()
)

FT_feP_grouped_r_general = (
    aggregate_by_food_group(
        FT_feP_grouped_r,
        group_map,
        group_level_name="Food_group"
    )
    .groupby(level=0)
    .sum()
)


In [45]:
FT_grouped_sum=(
    pd.concat([
        FT_lu_grouped_r_general,
        FT_wu_grouped_r_general,
        FT_feN_grouped_r_general,
        FT_feP_grouped_r_general
    ],axis=0, keys=['LU','WU','FE-N','FE-P']).T
)

In [46]:
FT_grouped_sum

LU                                         \
Food_group              Beef_lamb_pig Dairy_foods     Eggs     Fish   Fruits   
EUregion Country                                                               
East     Bulgaria            1.56e-06    2.96e-06 1.26e-07 2.87e-08 2.87e-07   
         Czech Republic      1.59e-06    1.34e-06 6.19e-08 1.90e-08 2.41e-07   
         Hungary             1.98e-06    1.68e-06 1.21e-07 1.45e-08 2.79e-07   
         Poland              1.34e-06    1.71e-06 1.19e-07 1.07e-08 2.43e-07   
         Romania             1.52e-06    2.68e-06 2.02e-07 1.59e-08 2.39e-07   
         Slovakia            1.91e-06    1.72e-06 8.46e-08 2.03e-08 2.65e-07   
North    Denmark             7.27e-07    1.14e-06 3.84e-08 1.99e-08 6.69e-07   
         Estonia             1.68e-06    2.04e-06 1.74e-07 1.14e-08 4.84e-07   
         Finland             1.14e-06    1.99e-06 9.76e-08 1.49e-08 7.50e-07   
         Ireland             1.23e-06    8.20e-07 7.57e-08 1.72e-08 3.57e-07   
         Latvia              2.48e-06    2.33e-06 1.10e-07 2.34e-08 4.47e-07   
         Lithuania           2.88e-06    2.83e-06 1.83e-07 2.61e-08 5.06e-07   
         Sweden              1.31e-06    1.45e-06 9.91e-08 1.79e-08 5.13e-07   
South    Croatia             1.08e-06    1.38e-06 9.60e-08 9.19e-09 2.62e-07   
         Cyprus              2.06e-06    2.26e-06 1.32e-07 3.00e-08 2.06e-07   
         Greece              1.68e-06    4.71e-06 2.56e-07 1.12e-08 2.59e-07   
         Italy               1.28e-06    2.20e-06 7.65e-08 3.43e-08 2.96e-07   
         Malta               1.80e-06    1.94e-06 8.41e-08 3.16e-08 3.68e-07   
         Portugal            2.36e-06    2.89e-06 1.17e-07 5.84e-08 3.47e-07   
         Slovenia            1.19e-06    8.52e-07 1.13e-07 3.41e-08 3.87e-07   
         Spain               1.12e-06    1.41e-06 9.28e-08 3.05e-08 3.41e-07   
West     Austria             7.41e-07    1.58e-06 7.06e-08 6.17e-09 4.22e-07   
         Belgium             1.08e-06    1.47e-06 4.46e-08 1.76e-08 3.01e-07   
         France              9.92e-07    2.30e-06 6.82e-08 1.94e-08 4.28e-07   
         Germany             6.28e-07    1.14e-06 2.83e-08 2.65e-09 5.67e-07   
         Luxembourg          1.35e-06    1.74e-06 4.24e-08 1.81e-08 3.60e-07   
         Netherlands         9.10e-07    1.67e-06 6.60e-08 9.00e-09 4.68e-07   

                                                                            \
Food_group              Lard_tallow_butter  Legumes     Nuts Palm_coco_oil   
EUregion Country                                                             
East     Bulgaria                 5.01e-07 3.09e-08 8.19e-09      2.19e-08   
         Czech Republic           2.88e-06 2.97e-09 1.34e-09      3.13e-08   
         Hungary                  7.98e-07 9.16e-09 3.94e-09      5.12e-08   
         Poland                   2.83e-06 5.97e-09 3.69e-09      3.50e-08   
         Romania                  5.23e-07 6.86e-09 3.17e-09      4.30e-08   
         Slovakia                 3.13e-06 2.85e-09 1.45e-09      3.19e-08   
North    Denmark                  5.94e-07 2.05e-09 6.30e-09      2.06e-07   
         Estonia                  5.62e-07 3.37e-09 1.83e-09      1.99e-08   
         Finland                  5.12e-07 4.06e-09 5.28e-09      1.63e-07   
         Ireland                  6.71e-07 9.71e-09 2.94e-09      1.11e-07   
         Latvia                   1.94e-06 4.54e-09 3.35e-09      0.00e+00   
         Lithuania                1.95e-06 3.88e-09 3.37e-09      1.82e-08   
         Sweden                   4.78e-07 5.67e-09 4.47e-09      5.08e-08   
South    Croatia                  2.06e-07 2.19e-09 1.76e-09      1.45e-08   
         Cyprus                   3.45e-07 0.00e+00 2.87e-09      1.92e-08   
         Greece                   1.15e-07 2.97e-09 1.88e-09      3.03e-08   
         Italy                    4.65e-07 3.11e-09 5.09e-09      1.13e-08   
         Malta                    3.43e-07 0.00e+00 3.02e-09      1.57e-08